[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1YrfMl6sRIl-vVptTF14QM708E7q3V4zR/view?usp=drive_link)

# RAG Evaluation – Full Dataset

This notebook demonstrates how to evaluate RAG pipelines when you have both model responses and the retrieved contexts. Floeval scores answer relevancy and faithfulness to the retrieved documents.

**Objectives**
- Install Floeval and configure credentials
- Load a RAG dataset with `user_input`, `llm_response`, and `contexts`
- Run `answer_relevancy` and `faithfulness` metrics
- Inspect aggregate and per-sample results

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
%pip install git+https://github.com/FloTorch/floeval.git@dev

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass
# LLM and API configuration

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("your-api-key")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

The following cell imports the evaluation components and the LLM configuration schema.

In [ ]:
from floeval import Evaluation, DatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 4. Configure the LLM

The LLM configuration is built using the constants defined above. RAG metrics require both the chat model and the embedding model.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 5. Load the RAG Dataset

A RAG dataset is loaded with `user_input`, `llm_response`, and `contexts` (retrieved documents) for each sample. The optional `ground_truth` field enables context_precision and context_recall metrics.

In [ ]:
dataset = DatasetLoader.from_samples(
    [
        {
            "user_input": "What is RAG?",
            "llm_response": "RAG stands for Retrieval-Augmented Generation.",
            "contexts": ["RAG combines retrieval with generation for grounded responses."],
            "ground_truth": "Retrieval-Augmented Generation",
        },
        {
            "user_input": "How does photosynthesis work?",
            "llm_response": "Photosynthesis converts sunlight into energy using chlorophyll.",
            "contexts": ["Plants use chlorophyll to capture light.", "Converts CO2 and water into glucose."],
            "ground_truth": "Converts light into chemical energy",
        },
    ],
    partial_dataset=False,
)
print(f"Dataset loaded: {len(dataset.samples)} samples")

## 6. Create and Run the Evaluation

The `answer_relevancy` and `faithfulness` metrics are used to evaluate RAG quality. The `faithfulness` metric uses `contexts` to verify grounding in the retrieved documents.

In [ ]:
evaluation = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy", "faithfulness"],
    default_provider="ragas",
)

results = evaluation.run()
print("Aggregate scores:", results.aggregate_scores)

## 7. Inspect Per-Sample Results

Each sample includes scores for both metrics. Faithfulness indicates how well the answer stays grounded in the retrieved context.

In [ ]:
for i, sr in enumerate(results.sample_results, start=1):
    print(f"Sample {i}: {sr['user_input'][:50]}...")
    for key, data in sr.get("metrics", {}).items():
        print(f"  {key}: score={data.get('score')}")

---

## Summary

**What we did**
- Loaded a RAG dataset with `user_input`, `llm_response`, `contexts`, and `ground_truth`
- Ran `answer_relevancy` and `faithfulness` via the RAGAS provider
- Inspected aggregate scores and per-sample metrics

**Takeaways**
- RAG datasets require `contexts` for faithfulness and retrieval metrics.
- `ground_truth` enables context_precision and context_recall.
- You can mix RAGAS and DeepEval metrics in the same run.